# Experiment A: fine-tune the quantized Qwen3.5-9B (NF4) with the 4B's recipe (SemEval 2027 Task 6, Task 2)

S1_9B (lr 1e-4, GPU 0) and S2_9B (lr 2e-4, GPU 1), each capped at the 4B's 870 steps (13,920 examples), time-budgeted.
Afterwards each adapter answers the first 100 dev items per track. Scoring and the judge run locally.
Download `exp_a_results.zip` from the Output tab. **Settings -> Accelerator -> GPU T4 x2, Internet on.**

In [ ]:
REPO_URL = "https://github.com/hassan-31x/mmcqa-experiments.git"
BRANCH = "main"
BUDGET_HOURS = 11.0
TRAIN_HOURS = 9.5
ROWS = 100
WORK = "/kaggle/working/results"

In [ ]:
import glob, os, subprocess, sys
CODE = "/kaggle/working/mmcqa-experiments"
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
!nvidia-smi -L
!pip install -q -r {CODE}/requirements/qwen.txt
!pip list 2>/dev/null | grep -i -E "^(torch|transformers|peft|bitsandbytes|accelerate|huggingface-hub|tokenizers) "

In [ ]:
try:                                   # optional: a Kaggle secret HF_TOKEN avoids anonymous download limits
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret: continuing with anonymous downloads")

In [ ]:
!cd {CODE} && python experiments/exp_a_finetune_9b/run_kaggle.py --workdir {WORK} --budget-hours {BUDGET_HOURS} --train-hours {TRAIN_HOURS} --rows {ROWS}

In [ ]:
import json, glob
for p in sorted(glob.glob(f"{WORK}/logs/sft_*_sizing.json")) + sorted(glob.glob(f"{WORK}/logs/sft_*_summary.json")):
    print(os.path.basename(p), json.dumps(json.load(open(p)), indent=1)[:900])
for p in sorted(glob.glob(f"{WORK}/logs/sft_*_failed.json")):
    print("FAILED:", p, json.load(open(p))["error"])
print(open(f"{WORK}/logs/exp_a_completion.json").read()[:3000])

In [ ]:
ZIPNAME = "exp_a_results.zip"

import shutil, pathlib
media = pathlib.Path(os.environ.get("MMCQA_MEDIA", "/kaggle/temp/mmcqa_media"))
for tmp in (WORK + "/mmcultureqa", "/kaggle/working/mmcqa_media", "/kaggle/working/mmcqa_envs", str(media)):
    shutil.rmtree(tmp, ignore_errors=True)
!cd /kaggle/working && zip -qr {ZIPNAME} results && ls -lh {ZIPNAME}